In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from transformers import BertTokenizer, BertModel


In [22]:
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    x = []
    for atom in mol.GetAtoms():
        x.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            atom.GetHybridization().real
        ])
    x = torch.tensor(x, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t()
    return Data(x=x, edge_index=edge_index)


In [23]:
def morgan_fp(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    return torch.tensor(fp, dtype=torch.float)


In [24]:
AA = 'ACDEFGHIKLMNPQRSTVWY'

def protein_aac(seq):
    seq = seq.upper()
    return torch.tensor([seq.count(a)/len(seq) for a in AA], dtype=torch.float)


In [36]:
class CPIDataset(Dataset):
    def __init__(self, file_path):
        self.data = []
        with open(file_path) as f:
            for line in f:
                smiles, protein, label = line.strip().split()
                self.data.append((smiles, protein, int(label)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        smiles, protein, label = self.data[idx]

        graph = smiles_to_graph(smiles)

        graph.fp = morgan_fp(smiles)              # [2048]
        graph.aac = protein_aac(protein)          # [20]
        graph.protein_seq = protein               # string
        graph.y = torch.tensor(label, dtype=torch.float)

        return graph


In [ ]:
# def collate_fn(batch):
#     graphs = []
#     fps, aacs, labels, seqs = [], [], [], []

#     for g in batch:
#         graphs.append(g)
#         fps.append(g.fp)
#         aacs.append(g.aac)
#         labels.append(g.y)
#         seqs.append(g.protein_seq)

#     batched_graph = Data.from_data_list(graphs)

#     return {
#         "graph": batched_graph,
#         "fp": torch.stack(fps),
#         "aac": torch.stack(aacs),
#         "seqs": seqs,
#         "y": torch.stack(labels)
#     }


In [27]:
class CompoundGNN(nn.Module):
    def __init__(self, node_dim=4, hidden=128):
        super().__init__()
        self.conv1 = GCNConv(node_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return global_mean_pool(x, batch)


In [ ]:
class ProteinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(
            "Rostlab/prot_bert", do_lower_case=False
        )
        self.model = BertModel.from_pretrained("Rostlab/prot_bert")

        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, sequences):
        sequences = [" ".join(list(s)) for s in sequences]
        inputs = self.tokenizer(
            sequences,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        

        outputs = self.model(**inputs)
        return outputs.last_hidden_state[:, 0, :]


In [29]:
class ProteinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(
            "Rostlab/prot_bert", do_lower_case=False
        )
        self.model = BertModel.from_pretrained("Rostlab/prot_bert")

        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, sequences):
        sequences = [" ".join(list(s)) for s in sequences]
        inputs = self.tokenizer(
            sequences,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        outputs = self.model(**inputs)
        return outputs.last_hidden_state[:, 0, :]


In [40]:
class HybridCPI(nn.Module):
    def __init__(self):
        super().__init__()
        self.gnn = CompoundGNN()
        self.prot = ProteinEncoder()

        fusion_dim = 128 + 2048 + 1024 + 20

        self.fc = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    # def forward(self, graph, fp, aac, seqs):
    #     gnn_emb = self.gnn(graph.x, graph.edge_index, graph.batch)
    #     prot_emb = self.prot(seqs)

    #     fusion = torch.cat([gnn_emb, fp, prot_emb, aac], dim=1)
    #     return self.fc(fusion).squeeze()
    
    def forward(self, graph, fp, aac, seqs):
        gnn_emb = self.gnn(graph.x, graph.edge_index, graph.batch)
        prot_emb = self.prot(seqs)

        fusion = torch.cat([gnn_emb, fp, prot_emb, aac], dim=1)
        return self.fc(fusion).squeeze()



In [41]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt)**self.gamma * bce).mean()


In [46]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0

    for batch in loader:

        print(batch.fp.shape)       # [B, 2048]
        print(batch.aac.shape)      # [B, 20]
        print(len(batch.protein_seq))  # B
        print(batch.x.shape)        # [total_atoms, 4]

        optimizer.zero_grad()

        batch = batch.to(device)

        preds = model(
            graph=batch,
            fp=batch.fp,
            aac=batch.aac,
            seqs=batch.protein_seq
        )

        loss = loss_fn(preds, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [43]:
device = "cpu"

In [44]:
dataset = CPIDataset("../dataset/b_cancer/original/data.txt")
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

model = HybridCPI().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
loss_fn = FocalLoss()





In [ ]:
from torch_geometric.loader import DataLoader

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)


In [47]:
for epoch in range(10):
    loss = train_epoch(model, loader, optimizer, loss_fn)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


[15:03:30] DEPRECATION WARNING: please use MorganGenerator
[15:03:30] DEPRECATION WARNING: please use MorganGenerator
[15:03:30] DEPRECATION WARNING: please use MorganGenerator
[15:03:30] DEPRECATION WARNING: please use MorganGenerator


torch.Size([8192])
torch.Size([80])
4
torch.Size([348, 4])


RuntimeError: Tensors must have same number of dimensions: got 2 and 1